# 🎨 Üretken Yapay Zekaya Giriş (Generative AI)

Hoş geldiniz! Bu notebook'ta, makinelerin nasıl yeni içerikler (metin, görsel) ürettiğini adım adım uygulamalarla inceleyeceğiz.
Bilgisayarlı görü modelleri var olanı anlamlandırırken (Ayırt Edici - Discriminative), üretken modeller (Generative) sıfırdan yepyeni veriler oluşturur.

> 💡 **Not:** Bu notebook'taki örnekler tamamen yerel bilgisayarınızda ve açık kaynaklı modellerle çalışacak şekilde tasarlanmıştır.

## 📦 Gereksinimleri İndir

In [ ]:
!pip install transformers diffusers torch accelerate matplotlib numpy

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Ortam Tespiti: Eğer sisteminizde uygun bir ekran kartı (GPU) varsa işlemler çok daha hızlı gerçekleşir.
# Yoksa İşlemci (CPU) üzerinden hesaplamalar yapılacaktır.
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Çalışma ortamı: {device}")

## ✍️ Bölüm 1: Metinden Metin Üretimi (Büyük Dil Modelleri - LLM)

Büyük Dil Modelleri (LLM), milyarlarca parametre ile metinler üzerinde akıl yürütme ve üretim yapar.
Tüm çalışma prensipleri **"Sıradaki kelime tahmini" (Next-Token Prediction)** üzerine kuruludur.

Aşağıda Türkçe metin üretebilen `ytu-ce-cosmos/turkish-gpt2` modelini yerel olarak indirip kullanacağız.

In [ ]:
model_id = "ytu-ce-cosmos/turkish-gpt2"
print(f"Model {model_id} yükleniyor... (İlk çalıştırmada biraz zaman alabilir)")

# Model ve Tokenizer'ı (Metni sayılara çeviren yapı) indirelim.
tokenizer = GPT2Tokenizer.from_pretrained(model_id)
model = GPT2LMHeadModel.from_pretrained(model_id).to(device)

# Çakışmaları ve uyarıları engellemek için modelin varsayılan ayarlarını temizliyoruz:
model.generation_config.max_length = None 
model.generation_config.pad_token_id = tokenizer.eos_token_id

# Üretimi test edelim
prompt = "Yapay zeka teknolojileri gelecekte hayatımızı"
print(f"\n▶ Girdi Prompt'u: '{prompt}'\n")

# Eski "pipeline" kullanımı arkada çakışmalara ve karışıklıklara yol açtığı için doğrudan model.generate fonksiyonunu kullanıyoruz.
girdi_tensoru = tokenizer(prompt, return_tensors="pt").to(device)
cikti = model.generate(**girdi_tensoru, max_new_tokens=50, no_repeat_ngram_size=2)

print("✅ Üretilen Metin:")
print(tokenizer.decode(cikti[0], skip_special_tokens=True))

## 🧠 Bölüm 2: Prompt Mühendisliği ve Üretim Parametreleri

Bir yapay zeka modelinin vereceği tepkiyi sadece algoritmalarla değil, onu nasıl yönlendirdiğimiz de (Prompt Engineering) belirler.
Bunun yanında modeli uyarırken değiştirdiğimiz **parametreler** de (örn. Temperature) niyetimizi netleştirir.

- **Temperature (Sıcaklık)**: Düşükse cevaplar mantıksal ve dar kapsamlı (örn 0.2), Yüksekse daha orijinal ve geniş vizyonludur (örn 0.9).

In [ ]:
prompt_2 = "Geleceğin şehirlerinde insanlar"
girdi_tensoru_2 = tokenizer(prompt_2, return_tensors="pt").to(device)

print(f"▶ Girdi Prompt'u: {prompt_2}\n")

print("--- Düşük Temperature: 0.2 (Öngörülebilir / Standard) ---")
cikti_dusuk = model.generate(**girdi_tensoru_2, max_new_tokens=50, temperature=0.2, top_p=0.9, do_sample=True, no_repeat_ngram_size=2)
print(tokenizer.decode(cikti_dusuk[0], skip_special_tokens=True))

print("\n--- Yüksek Temperature: 0.9 (Şaşırtıcı) ---")
cikti_yuksek = model.generate(**girdi_tensoru_2, max_new_tokens=50, temperature=0.9, top_p=0.9, do_sample=True, no_repeat_ngram_size=2)
print(tokenizer.decode(cikti_yuksek[0], skip_special_tokens=True))

## 🌄 Bölüm 3: Metinden Görüntü Üretimi (Diffusion Modelleri)

Görüntü üretiminde **Diffusion Modelleri** (Stable Diffusion vb.) öne çıkar. 
Çalışma mantığı temelde **Gürültü Temizleme (Denoising)** işlemine dayanır:
1. Tamamen rastgele bir kumlama (gürültü) ile başlanır.
2. Metin girdisine (örn: 'kırmızı bir araba') göre adım adım (Sampling steps) bu gürültü, aranan görsele dönüşene dek yontulur.

Aşağıdaki örnekte yerel makinede çalışabilmesi şartıyla küçültülmüş bir Stable Diffusion versiyonunu indiriyoruz.

In [ ]:
print("Diffusion modeli yükleniyor... (İlk indiğinde ~2GB yaklaşık internet indirmesi yapabilir)")

# Bellekte daha az yer kaplaması için GPU varsa yarım hassasiyet kullanılır
dtype = torch.float16 if device in ["cuda", "mps"] else torch.float32

# Orijinal Stable Diffusion 1.5'u kullanabiliriz
model_str = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_str, torch_dtype=dtype)
pipe = pipe.to(device)

# Slaytlardaki gibi "Detaylı Prompt" örneğini kullanıyoruz (Şimdilik İngilizce ağırlıklı anlarlar)
prompt_en = "A futuristic cyberpunk city with neon signs, highly detailed, digital art, 8k resolution"
print(f"\n▶ Prompt: {prompt_en}\n")

print("⏳ Görüntü üretiliyor... (CPU kullanan mimariler için biraz zaman alabilir)")

# num_inference_steps: Gürültü temizleme adımı (slaytlardaki gürültü adım sayısı tablosu!)
# guidance_scale: Prompt'a bağlılık (CFG Scale)
gorsel = pipe(prompt_en, num_inference_steps=20, guidance_scale=7.5).images[0]

# Üretilen resmi çizdir
plt.figure(figsize=(10, 10))
plt.imshow(np.array(gorsel))
plt.title(f"Üretken Görüntü (Stable Diffusion)\nCFG: 7.5 | Steps: 20")
plt.axis('off')
plt.show()